In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
from bs4 import BeautifulSoup

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier

from sklearn.model_selection import (
    StratifiedKFold,
    GridSearchCV
)

print("P05 model comparison environment ready!")

P05 model comparison environment ready!


In [2]:
TRAIN_PATH = Path("../data/processed/train.csv")

train_df = pd.read_csv(TRAIN_PATH)

X_train = train_df["review"]
y_train = train_df["sentiment"]

print("Training samples:", len(X_train))
print("\nClass distribution:")
print(y_train.value_counts())

Training samples: 40006

Class distribution:
sentiment
positive    20004
negative    20002
Name: count, dtype: int64


In [3]:
def strip_html_batch(texts):
    return [
        BeautifulSoup(text, "html.parser").get_text(separator=" ")
        for text in texts
    ]

print("HTML cleaning function ready!")

HTML cleaning function ready!


In [4]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

print("Five-fold stratified cross-validation ready!")

Five-fold stratified cross-validation ready!


In [5]:
# P05: Model comparison pipeline

from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import FunctionTransformer

# Reusable text-classification pipeline
def strip_html_batch(texts):
    return [
        BeautifulSoup(text, "html.parser").get_text(separator=" ")
        for text in texts
    ]

pipeline = Pipeline([
    (
        "html",
        FunctionTransformer(
            strip_html_batch,
            validate=False
        )
    ),
    (
        "tfidf",
        TfidfVectorizer(
            ngram_range=(1, 2),
            min_df=2,
            max_features=100000
        )
    ),
    ("classifier", LogisticRegression(max_iter=1000))
])

print("Model-tuning pipeline created!")

Model-tuning pipeline created!


In [6]:
# Tune Logistic Regression using 5-fold cross-validation

lr_params = {
    "classifier__C": [0.1, 1, 10]
}

lr_search = GridSearchCV(
    estimator=pipeline,
    param_grid=lr_params,
    cv=cv,
    scoring="accuracy",
    n_jobs=-1,
    verbose=2
)

lr_search.fit(X_train, y_train)

print("\nBest Logistic Regression parameters:")
print(lr_search.best_params_)

print(f"Best CV accuracy: {lr_search.best_score_:.4f}")

Fitting 5 folds for each of 3 candidates, totalling 15 fits
[CV] END ..................................classifier__C=0.1; total time=  40.9s
[CV] END ..................................classifier__C=0.1; total time=  41.4s
[CV] END ..................................classifier__C=0.1; total time=  40.9s
[CV] END ..................................classifier__C=0.1; total time=  40.6s
[CV] END ..................................classifier__C=0.1; total time=  39.6s
[CV] END ....................................classifier__C=1; total time=  40.0s
[CV] END ....................................classifier__C=1; total time=  39.6s
[CV] END ....................................classifier__C=1; total time=  39.5s
[CV] END ....................................classifier__C=1; total time=  39.8s
[CV] END ....................................classifier__C=1; total time=  40.0s
[CV] END ...................................classifier__C=10; total time=  39.6s
[CV] END ...................................class

In [7]:
# P05: Tune Multinomial Naive Bayes

from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import FunctionTransformer

nb_pipeline = Pipeline([
    (
        "html",
        FunctionTransformer(
            strip_html_batch,
            validate=False
        )
    ),
    (
        "tfidf",
        TfidfVectorizer(
            ngram_range=(1, 2),
            min_df=2,
            max_features=100000
        )
    ),
    ("classifier", MultinomialNB())
])

nb_params = {
    "classifier__alpha": [0.1, 0.5, 1.0, 2.0]
}

nb_search = GridSearchCV(
    estimator=nb_pipeline,
    param_grid=nb_params,
    cv=cv,
    scoring="accuracy",
    n_jobs=-1,
    verbose=2
)

nb_search.fit(X_train, y_train)

print("\nBest Naive Bayes parameters:")
print(nb_search.best_params_)

print(f"Best CV accuracy: {nb_search.best_score_:.4f}")

Fitting 5 folds for each of 4 candidates, totalling 20 fits
[CV] END ..............................classifier__alpha=0.1; total time=  39.0s
[CV] END ..............................classifier__alpha=0.1; total time=  39.3s
[CV] END ..............................classifier__alpha=0.1; total time=  38.1s
[CV] END ..............................classifier__alpha=0.1; total time=  39.1s
[CV] END ..............................classifier__alpha=0.1; total time=  38.2s
[CV] END ..............................classifier__alpha=0.5; total time=  38.1s
[CV] END ..............................classifier__alpha=0.5; total time=  38.2s
[CV] END ..............................classifier__alpha=0.5; total time=  38.6s
[CV] END ..............................classifier__alpha=0.5; total time=  38.7s
[CV] END ..............................classifier__alpha=0.5; total time=  38.4s
[CV] END ..............................classifier__alpha=1.0; total time=  38.3s
[CV] END ..............................classifier

In [8]:
# P05: Tune Linear SVM

from sklearn.svm import LinearSVC
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import FunctionTransformer

svm_pipeline = Pipeline([
    (
        "html",
        FunctionTransformer(
            strip_html_batch,
            validate=False
        )
    ),
    (
        "tfidf",
        TfidfVectorizer(
            ngram_range=(1, 2),
            min_df=2,
            max_features=100000
        )
    ),
    (
        "classifier",
        LinearSVC(random_state=42, max_iter=3000)
    )
])

svm_params = {
    "classifier__C": [0.1, 1, 10]
}

svm_search = GridSearchCV(
    estimator=svm_pipeline,
    param_grid=svm_params,
    cv=cv,
    scoring="accuracy",
    n_jobs=-1,
    verbose=2
)

svm_search.fit(X_train, y_train)

print("\nBest Linear SVM parameters:")
print(svm_search.best_params_)

print(f"Best CV accuracy: {svm_search.best_score_:.4f}")

Fitting 5 folds for each of 3 candidates, totalling 15 fits
[CV] END ..................................classifier__C=0.1; total time=  39.5s
[CV] END ..................................classifier__C=0.1; total time=  39.6s
[CV] END ..................................classifier__C=0.1; total time=  39.4s
[CV] END ..................................classifier__C=0.1; total time=  39.8s
[CV] END ..................................classifier__C=0.1; total time=  40.1s
[CV] END ....................................classifier__C=1; total time=  40.4s
[CV] END ....................................classifier__C=1; total time=  39.8s
[CV] END ....................................classifier__C=1; total time=  40.3s
[CV] END ....................................classifier__C=1; total time=  40.1s
[CV] END ....................................classifier__C=1; total time=  39.8s
[CV] END ...................................classifier__C=10; total time=  44.1s
[CV] END ...................................class

In [9]:
rf_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        ngram_range=(1, 1),
        min_df=10,
        max_features=2000,
        dtype=np.float32
    )),
    ("classifier", RandomForestClassifier(
        n_estimators=20,
        max_depth=15,
        random_state=42,
        n_jobs=-1
    ))
])

rf_search = GridSearchCV(
    rf_pipeline,
    {"classifier__max_depth": [15]},
    cv=cv,
    scoring="accuracy",
    n_jobs=1,
    verbose=1
)

rf_search.fit(X_train, y_train)

print("Best parameters:", rf_search.best_params_)
print("CV accuracy:", rf_search.best_score_)

Fitting 5 folds for each of 1 candidates, totalling 5 fits
Best parameters: {'classifier__max_depth': 15}
CV accuracy: 0.8043294232060033


In [10]:
# Consolidate cross-validation results

model_results = pd.DataFrame([
    {
        "Model": "Logistic Regression",
        "Best Parameters": str(lr_search.best_params_),
        "CV Accuracy": lr_search.best_score_
    },
    {
        "Model": "Multinomial Naive Bayes",
        "Best Parameters": str(nb_search.best_params_),
        "CV Accuracy": nb_search.best_score_
    },
    {
        "Model": "Linear SVM",
        "Best Parameters": str(svm_search.best_params_),
        "CV Accuracy": svm_search.best_score_
    },
    {
        "Model": "Random Forest",
        "Best Parameters": str(rf_search.best_params_),
        "CV Accuracy": rf_search.best_score_
    }
])

model_results = model_results.sort_values(
    "CV Accuracy",
    ascending=False
).reset_index(drop=True)

model_results["CV Accuracy (%)"] = (
    model_results["CV Accuracy"] * 100
).round(2)

# Display results
display(model_results)

# Save results
RESULTS_DIR = Path("../results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

model_results.to_csv(
    RESULTS_DIR / "model_comparison.csv",
    index=False
)

print("Model comparison saved successfully.")

,Model,Best Parameters,CV Accuracy,CV Accuracy (%)
0,Linear SVM,{'classifier__C': 1},0.914013,91.40
1,Logistic Regression,{'classifier__C': 10},0.912813,91.28
2,Multinomial Naive Bayes,{'classifier__alpha': 0.1},0.889167,88.92
3,Random Forest,{'classifier__max_depth': 15},0.804329,80.43


Model comparison saved successfully.


## Full cross-validation tables

The summary table above keeps only the best candidate per model. This cell
saves every candidate with its fold-to-fold standard deviation and fit time,
so the choice of hyperparameters can be inspected afterwards.

In [11]:
cv_tables = []
for model_name, search in {
    "Logistic Regression": lr_search,
    "Multinomial Naive Bayes": nb_search,
    "Linear SVM": svm_search,
    "Random Forest": rf_search,
}.items():
    table = pd.DataFrame(search.cv_results_)[
        ["params", "mean_test_score", "std_test_score",
         "mean_fit_time", "rank_test_score"]
    ].copy()
    table.insert(0, "model", model_name)
    table["params"] = table["params"].astype(str)
    cv_tables.append(table)

cv_summary = pd.concat(cv_tables, ignore_index=True)
cv_summary.to_csv(RESULTS_DIR / "cv_results.csv", index=False)
display(cv_summary.round(4))

,model,params,mean_test_score,std_test_score,mean_fit_time,rank_test_score
0,Logistic Regression,{'classifier__C': 0.1},0.8647,0.0045,34.3500,3
1,Logistic Regression,{'classifier__C': 1},0.9027,0.0009,33.5716,2
2,Logistic Regression,{'classifier__C': 10},0.9128,0.0019,30.9724,1
3,Multinomial Naive Bayes,{'classifier__alpha': 0.1},0.8892,0.0014,32.6605,1
4,Multinomial Naive Bayes,{'classifier__alpha': 0.5},0.8871,0.0015,32.4645,2
5,Multinomial Naive Bayes,{'classifier__alpha': 1.0},0.8853,0.0011,32.5700,3
6,Multinomial Naive Bayes,{'classifier__alpha': 2.0},0.8823,0.0013,32.7097,4
7,Linear SVM,{'classifier__C': 0.1},0.9057,0.0011,33.4438,3
8,Linear SVM,{'classifier__C': 1},0.9140,0.0025,34.0175,1
9,Linear SVM,{'classifier__C': 10},0.9075,0.0018,35.8632,2
